In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import decimal as dec
import re
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [2]:
movies = pd.read_csv('datasets/IMDb movies.csv', sep=";", encoding="latin1")
moviesaux = pd.read_csv('datasets/IMDb movies.csv', sep=";", encoding="latin1")
ratings = pd.read_csv('datasets/IMDb ratings.csv', sep=";")
ratingsaux = pd.read_csv('datasets/IMDb ratings.csv', sep=";")


In [3]:
#Parte da remoção de colunas do Pedro

missing_values_m = moviesaux.isnull().sum()
# print('MOVEIS', missing_values_m)

tamanho = len(movies.index)

moviesaux = moviesaux.drop(['imdb_title_id'], axis=1)

missing_values = movies.isnull().sum()

percentagem = (missing_values / tamanho) * 100

df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentagem': percentagem.round(2)
})

#print(df.to_string())
indice = df[df['Percentagem']> 50 ].index
movies = movies.drop(columns=indice)

#---------------------------------------------------------------

tamanho = len(ratings.index)

ratingsaux = ratingsaux.drop(['imdb_title_id'], axis=1)

missing_values = ratings.isnull().sum()

percentagem = (missing_values / tamanho) * 100

df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentagem': percentagem.round(2)
})

indice = df[df['Percentagem']> 50 ].index
df.drop(indice, inplace=True) 

#print(df.to_string())

dados = ratings[~ratings.isna()]
colunas = [
    "allgenders_0age_votes", "allgenders_18age_votes",
    "allgenders_30age_votes", "allgenders_45age_votes",
    "males_allages_votes", "males_0age_votes", "males_18age_votes",
    "males_30age_votes", "males_45age_votes",
    "females_allages_votes", "females_0age_votes", "females_18age_votes",
    "females_30age_votes", "females_45age_votes",
    "top1000_voters_votes", "us_voters_votes", "non_us_voters_votes"
]

votos_total = dados['total_votes'].sum()
colunas_a_remover = []

for coluna in colunas:
    votos_demografia = dados[coluna].sum()
    proporcao =  votos_demografia / votos_total
    #print('Proporção de votos em percentagem da coluna', coluna, proporcao.round(2)*100)
    if proporcao < .10:
        colunas_a_remover.append(coluna)
    proporcao = 0

ratings = ratings.drop(columns=indice)

In [5]:
dataset = pd.merge(movies, ratings, on="imdb_title_id", how="inner")
pd.set_option('display.max_columns', None)  #Mostra todas as colunas
#display(dataset.head(5))

In [6]:
#---------------------------------------------------------------------------------------------------------------------------------------
#CRIAÇÃO DE UTILIZADORES
#---------------------------------------------------------------------------------------------------------------------------------------

users = list(range(1, 101)) #Criar 100 users
user_ratings = []
num_filmesfixos = 1250
num_filmestotais = 2500

#Escolher alguns filmes, para que haja garantia que temos mais do que uma review em maior parte dos filme.
#Desta forma reduz-se o número de zeros na matriz de similariedade
filmes_fixos = np.random.choice(dataset['imdb_title_id'], size = num_filmesfixos, replace = False)

for user in users:
    filmes_avaliados = list(filmes_fixos)

    #Preencher o resto com filmes aleatórios
    filmes_restantes = np.random.choice(
        dataset[~dataset['imdb_title_id'].isin(filmes_fixos)]['imdb_title_id'],
        size = num_filmestotais - num_filmesfixos, 
        replace = False
    )

    filmes_avaliados.extend(filmes_restantes)

    #Gerar ratings para cada filme avaliado pelo utilizador
    for movie_id in filmes_avaliados:
        avg_vote = dataset.loc[dataset['imdb_title_id'] == movie_id, 'avg_vote'].values[0]
        rating = np.clip(np.random.normal(avg_vote, 3), 1, 10) #Centro da distribuição é o avg_vote e a standard deviation é 3
        user_ratings.append({' User ID ': user, ' Movie ID ': movie_id, ' User Rating ': round(rating, 1)})


ratings_df = pd.DataFrame(user_ratings)
ratings_df.to_csv("user_ratings.csv", index=False) 
#print(ratings_df)



In [7]:
ratings_df = pd.DataFrame(user_ratings)
ratings_df.to_csv("user_ratings.csv", index=False) 
#print(ratings_df)

#Função auxiliar para conseguir o nome do filme
def get_titulo(movie_id):
    title = dataset.loc[dataset['imdb_title_id'] == movie_id, 'original_title']
    return title.values[0]

#Função auxiliar para conseguir o genero preferido
def genero_preferido(user_id, ratings_df, dataset):
    filmes_user = ratings_df[ratings_df[' User ID '] == user_id]
    filmes_user = filmes_user.merge(dataset[['imdb_title_id', 'genre']], left_on=' Movie ID ', right_on='imdb_title_id')

    #Expandir géneros (caso estejam separados por vírgulas)
    filmes_user['genre'] = filmes_user['genre'].str.split(',')
    filmes_user = filmes_user.explode('genre')
    filmes_user['genre'] = filmes_user['genre'].str.strip()

    #Calcular média de rating por género
    media_por_genero = filmes_user.groupby('genre')[' User Rating '].mean().sort_values(ascending=False)
    top_generos = media_por_genero.index[:2].tolist()  
    return top_generos

In [7]:
#---------------------------------------------------------------------------------------------------------------------------------------
#ITEM-BASED COLLABORATIVE FILTERING
#---------------------------------------------------------------------------------------------------------------------------------------

user_ratings = pd.read_csv("user_ratings.csv") 

#Criar matriz utilizador-filme
user_movie_matrix = user_ratings.pivot(index = ' User ID ', columns = ' Movie ID ', values = ' User Rating ')
user_movie_matrix = user_movie_matrix.fillna(0)
#print(user_movie_matrix)

#Calcular similaridade entre filmes através de Cosine Similarity
item_similarity = cosine_similarity(user_movie_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)
#print(item_similarity_df.head())

#Função para recomendar filmes (IBC - Item-Based Collaborative)
def recomendar_IBC(movie_id, num_rec=5):
    if movie_id not in item_similarity_df.index:
        print("Filme não encontrado na base de dados!")
        return []
    
    similares_item = item_similarity_df[movie_id].sort_values(ascending=False)[1:num_rec+1]  
    recomendacoes_item = [get_titulo(filme) for filme in similares_item.index]  
    return recomendacoes_item

#Teste de recomendações
filme_teste = user_ratings[' Movie ID '].sample(1).values[0]
nome= get_titulo(filme_teste)
print(f"Recomendações para quem viu '{nome}': {recomendar_IBC(filme_teste)}")

#Ainda falta melhorar o cálculo das recomendações, apenas procura filmes com ratings parecidos, as recomendações não são as melhores

Recomendações para quem viu 'Dial a Prayer': ['Joan of Paris', 'Eleven Days, Eleven Nights 2', "You're Killing Me...", 'Paradas contínuas', 'Within']


In [25]:
#---------------------------------------------------------------------------------------------------------------------------------------
#USER-BASED COLLABORATIVE FILTERING 
#---------------------------------------------------------------------------------------------------------------------------------------

user_ratings = pd.read_csv("user_ratings.csv") 

#user_movie_matrix = user_ratings.pivot(index = ' User ID ', columns = ' Movie ID ', values = ' User Rating ')
#user_movie_matrix = user_movie_matrix.fillna(0)
#print(user_movie_matrix)

train_data = {}
test_data = {}
sem_ratings = []
test_size = 0.6


#Separar os dados de cada utilizador em treino e teste
for user_id in user_ratings[' User ID '].unique():
    filmes_avaliados = user_ratings[user_ratings[' User ID '] == user_id]

    if len(filmes_avaliados) > 5:
        train, test = train_test_split(filmes_avaliados, test_size=test_size, random_state=4465)
        train_data[user_id] = train
        test_data[user_id] = test
    else:
        sem_ratings.append(user_id)

#Criar nova matriz utilizador-filme apenas com os dados de treino
train_ratings = pd.concat(train_data.values())  # Reunir todas as avaliações de treino
train_movie_matrix = train_ratings.pivot(index=' User ID ', columns=' Movie ID ', values=' User Rating ').fillna(0)

#Calcular similaridade entre filmes
user_similarity = cosine_similarity(train_movie_matrix)
#user_similarity = train_movie_matrix.T.corr(method='pearson')
user_similarity_df = pd.DataFrame(user_similarity, index=train_movie_matrix.index, columns=train_movie_matrix.index)
#print(user_similarity_df.head())

def rerank_diversidade_novidade(recs, dataset):
    df = dataset[dataset['imdb_title_id'].isin(recs)].copy()
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df['votes'] = pd.to_numeric(df.get('votes', 0), errors='coerce')

    #Score novidade: quanto mais recente, maior o score
    ano_max = df['year'].max()
    ano_min = df['year'].min()
    df['score_novidade'] = (df['year'] - ano_min) / (ano_max - ano_min + 1e-9)

    #Score popularidade invertida: quanto menos votos, maior o score
    vot_max = df['votes'].max()
    df['score_popularidade'] = 1 - (df['votes'] / (vot_max + 1e-9))

    #Combinações dos dois scores
    df['score_rerank'] = 0.6 * df['score_novidade'] + 0.4 * df['score_popularidade']

    #Ordena segundo este novo score e devolve a lista de IDs
    return df.sort_values('score_rerank', ascending=False)['imdb_title_id'].tolist()


def recomendar_UBC(user_id, num_rec, dataset, ratings_df):

    sims = user_similarity_df[user_id].drop(user_id).sort_values(ascending=False)
    top_k = sims[:20]

    # Score inicial
    weighted_ratings = (train_movie_matrix.loc[top_k.index].T.dot(top_k) / top_k.sum())

    # Filtra filmes já vistos
    user_rated = train_movie_matrix.loc[user_id]
    preds = weighted_ratings[user_rated == 0]

    # Obter géneros preferidos
    generos_top = genero_preferido(user_id, ratings_df, dataset)

    # Obter géneros dos filmes recomendados
    filmes_generos = dataset.set_index('imdb_title_id').loc[preds.index][['genre']]
    filmes_generos['genre'] = filmes_generos['genre'].fillna('').str.split(',')
    filmes_generos = filmes_generos['genre'].apply(lambda genres: [g.strip() for g in genres])

    # Aplicar peso dos géneros diretamente ao score
    genre_weights = []
    for generos in filmes_generos:
        if any(g in generos_top for g in generos):
            genre_weights.append(1.5)
        else:
            genre_weights.append(0.7)
    
    # Combina os scores dos vizinhos com os pesos dos géneros
    preds = preds * genre_weights
    preds = preds.sort_values(ascending=False)

    # Remove a metade inferior
    half = int(len(preds) * 0.5)
    if half > 0:
        preds = preds.iloc[:-half]

    #sampled = preds.sample(100, weights=preds, random_state=42)
    #sampled_ids = sampled.index.tolist()
    sampled_ids = preds.index[:50].tolist()

    reranked = rerank_diversidade_novidade(sampled_ids, dataset)

    return reranked[:num_rec]
    #return preds.index[:num_rec].tolist()

In [26]:
popular_movies = (train_ratings.groupby(' Movie ID ')[' User Rating '].mean().sort_values(ascending=False).index.tolist())

def fallback_recomendacoes(user_id, dataset, ratings_df):
    #Filmes já vistos
    vistos = train_movie_matrix.loc[user_id]
    filmes_vistos = set(vistos[vistos > 0].index)

    #Géneros preferidos
    generos = genero_preferido(user_id, ratings_df, dataset)

    #Filmes do género preferido que o user ainda não viu
    dataset_genero = dataset.copy()
    dataset_genero['genre'] = dataset_genero['genre'].fillna('').str.split(',')
    dataset_genero = dataset_genero.explode('genre')
    dataset_genero['genre'] = dataset_genero['genre'].str.strip()

    candidatos_genero = dataset_genero[
        dataset_genero['genre'].isin(generos)
    ].drop_duplicates(subset='imdb_title_id')

    candidatos_genero = candidatos_genero[
        ~candidatos_genero['imdb_title_id'].isin(filmes_vistos)
    ]

    num_generos = np.random.randint(4,8)
    num_populares = 10 - num_generos
    filmes_genero = candidatos_genero['imdb_title_id'].sample(
        min(num_generos, len(candidatos_genero)), random_state=4465).tolist()

    #Filmes populares ou recentes ainda não vistos
    candidatos_extras = [f for f in popular_movies if f not in filmes_vistos]
    filmes_extras = candidatos_extras[:num_populares]

    return filmes_genero + filmes_extras



def avaliar_recomendacoes_com_genero(user_id, recomendacoes, ground_truth, dataset, genero_peso=0.5, filme_peso=0.5):
    filmes_certos = sum(1 for filme in recomendacoes if filme in ground_truth)
    
    generos_user = genero_preferido(user_id, ratings_df, dataset)

    #Pega os géneros dos filmes recomendados
    filmes_rec_genero = dataset[dataset['imdb_title_id'].isin(recomendacoes)][['imdb_title_id', 'genre']]
    filmes_rec_genero['genre'] = filmes_rec_genero['genre'].str.split(',')
    filmes_rec_genero = filmes_rec_genero.explode('genre')
    filmes_rec_genero['genre'] = filmes_rec_genero['genre'].str.strip()

    #Ver quantos dos filmes recomendados contêm o género preferido
    genero_certo = filmes_rec_genero['genre'].isin(generos_user).sum()

    total_recs = len(recomendacoes)
    filme_score = filmes_certos / total_recs
    genero_score = genero_certo / total_recs

    score_final = (filme_peso * filme_score) + (genero_peso * genero_score)
    return score_final


#Teste para todos os users no dataset que tem ratings suficientes
scores_array = []
for user_id in test_data.keys():
    ground_truth = test_data[user_id][' Movie ID '].tolist()  
    recomendacoes = recomendar_UBC(user_id, 10, dataset, ratings_df)  
    score = avaliar_recomendacoes_com_genero(user_id, recomendacoes, ground_truth, dataset)
    if score < 0.1:
        recomendacoes = fallback_recomendacoes(user_id, dataset, ratings_df)
        score = avaliar_recomendacoes_com_genero(user_id, recomendacoes, ground_truth, dataset)
    scores_array.append(score)
print([round(float(score), 3) for score in scores_array])
print(f"Avaliação Média: {np.average(scores_array):.2f}")


#Demonstração da cold start para users sem ratings suficientes
#filmes_novos = dataset.sort_values(by='year', ascending=False).head(5)
#ids = filmes_novos['imdb_title_id'].tolist()
#print([get_titulo(imdb_id) for imdb_id in ids])

#user_id = np.random.randint(1,100)
#ground_truth = test_data[user_id][' Movie ID '].tolist()  
#recomendacoes = recomendar_UBC(user_id, 10, dataset, ratings_df)  
#score = avaliar_recomendacoes_com_genero(user_id, recomendacoes, ground_truth, dataset)
#print(f"Avaliação: {score}")



[0.5, 0.55, 0.45, 0.5, 0.5, 0.55, 0.45, 0.4, 0.5, 0.5, 0.4, 0.5, 0.4, 0.45, 0.45, 0.5, 0.45, 0.45, 0.45, 0.6, 0.45, 0.45, 0.5, 0.55, 0.35, 0.4, 0.5, 0.45, 0.4, 0.5, 0.45, 0.45, 0.5, 0.4, 0.2, 0.3, 0.4, 0.45, 0.35, 0.5, 0.45, 0.4, 0.45, 0.2, 0.4, 0.55, 0.45, 0.45, 0.4, 0.5, 0.5, 0.45, 0.4, 0.45, 0.5, 0.4, 0.5, 0.4, 0.5, 0.4, 0.55, 0.5, 0.4, 0.5, 0.45, 0.5, 0.5, 0.35, 0.45, 0.5, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.55, 0.5, 0.4, 0.55, 0.45, 0.45, 0.45, 0.6, 0.5, 0.5, 0.45, 0.4, 0.4, 0.5, 0.5, 0.4, 0.5, 0.45, 0.5, 0.5, 0.45, 0.45, 0.4, 0.4]
Avaliação Média: 0.46
